# Day 9: Snowflake Security, Dynamic Masking & dbt Gold Marts

Simulate Gold metric calculations (30-day readmissions) and role-based column masking on sample records using DuckDB.

In [ ]:
import duckdb
import pandas as pd

conn = duckdb.connect(':memory:')

conn.execute("""
CREATE TABLE visit_occurrence (
    person_id INT,
    visit_occurrence_id INT,
    visit_concept_id INT,
    visit_start_date DATE,
    visit_end_date DATE
);
""")

conn.execute("""
INSERT INTO visit_occurrence VALUES 
(1, 101, 9201, '2023-01-01', '2023-01-05'), -- Inpatient, discharge Jan 5
(1, 102, 9201, '2023-01-20', '2023-01-22'), -- Readmission within 30 days (15 days later)
(2, 201, 9201, '2023-02-01', '2023-02-10'), -- Inpatient
(2, 202, 9201, '2023-04-01', '2023-04-05'); -- Not a 30-day readmission
""")
print("Silver visit_occurrence table created.")

## 1. Simulate dbt Gold Mart: 30-Day Readmissions

In [ ]:
# DBT logic translated to standard SQL for DuckDB
gold_query = """
WITH discharges AS (
    SELECT 
        person_id,
        visit_occurrence_id,
        visit_end_date as discharge_date
    FROM visit_occurrence
    WHERE visit_concept_id = 9201
),
readmissions AS (
    SELECT 
        d1.person_id,
        d1.visit_occurrence_id as index_visit_id,
        d2.visit_occurrence_id as readmission_visit_id
    FROM discharges d1
    JOIN visit_occurrence d2
        ON d1.person_id = d2.person_id
        AND d2.visit_start_date > d1.discharge_date
        AND datediff('day', d1.discharge_date, d2.visit_start_date) <= 30
        AND d2.visit_concept_id = 9201
)
SELECT 
    v.person_id,
    COUNT(DISTINCT v.visit_occurrence_id) as total_index_admissions,
    COUNT(DISTINCT r.readmission_visit_id) as total_30d_readmissions,
    CAST(COUNT(DISTINCT r.readmission_visit_id) AS FLOAT) / NULLIF(COUNT(DISTINCT v.visit_occurrence_id), 0) as readmission_rate
FROM visit_occurrence v
LEFT JOIN readmissions r ON v.person_id = r.person_id
WHERE v.visit_concept_id = 9201
GROUP BY v.person_id;
"""

print("Gold 30-day Readmission Metrics:")
print(conn.execute(gold_query).df())

## 2. Simulate Dynamic Data Masking

In [ ]:
# Simulate a table with PII
conn.execute("""
CREATE TABLE patient_pii (
    person_id INT,
    ssn VARCHAR,
    dob DATE
);
INSERT INTO patient_pii VALUES 
(1, '123-45-6789', '1980-05-15'),
(2, '987-65-4321', '1992-11-20');
""")

# Simulate Analyst Role View (Masked)
analyst_view_query = """
SELECT 
    person_id,
    '***-**-****' as ssn, -- Masked SSN
    DATE_TRUNC('year', dob) as dob -- Generalize DOB to year
FROM patient_pii;
"""

print("View from ANALYST_MASKED Role:")
print(conn.execute(analyst_view_query).df())